In [1]:
import json
import os.path
import torch
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pickle
from shapely.geometry import Point
import scienceplots
plt.style.use(['science', "no-latex"])
import random

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)


In [2]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

In [3]:
# seed = random.randint(0, 10000)
seed = 55
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Read dataset

In [4]:
ITL3_region = gpd.read_file(r"./results/intermediate/ITL3_region.gpkg")
substations = gpd.read_file(r"./results/intermediate/substations.gpkg")

# Determine the region

In [5]:
interested_locations = {
    # "London": {},
    "TLI3": {},
    "TLI4": {},
    "TLI5": {},
    "TLI6": {},
    "TLI7": {},
    "TLH1": {},
    "TLH2": {},
    "TLH3": {},
    "TLJ1":{},
    "TLF1":{},
    "TLF2":{},
    "TLC1":{},
    "TLC2":{},
    "TLD3":{},
    "TLD4":{},
    "TLD6":{},
    "TLG1":{},
    "TLG2":{},
    "TLE3":{},
    "TLE4":{},
}

In [6]:
for itl2_code in interested_locations.keys():
    interested_region = ITL3_region[ITL3_region['ITL2'] == itl2_code]
    interested_substations = substations[substations['ITL2'] == itl2_code]

    interested_locations[itl2_code] = {
        "region": interested_region.reset_index(),
        "substation": interested_substations.reset_index()
    }

# Grid points

In [7]:
from SpatialAllocation.GNN.utils.GraphBuilder import prepare_hetero_graph_from_processed, preprocess_features
from sklearn.preprocessing import StandardScaler

In [8]:
with open(f"./results/intermediate/GridPoint_GNN/all_node_features_col.pickle", "rb") as f:
    numerical_col_names_all, categorical_col_members_all = pickle.load(f)

In [9]:
for key in interested_locations.keys():
    graph_data_path = f"./results/intermediate/GNN/{key}_prepared_graph_data.pt"
    if os.path.exists(graph_data_path):
        print(f"Graph data for {key} already exists, skipping preparation.")
        continue
    with open(f"./results/intermediate/GridPoint/{key}_grid_points.pickle", "rb") as f:
        grid_gdf, step_size_m = pickle.load(f)
    gdf_a = grid_gdf.to_crs("EPSG:3857").copy()
    gdf_t = interested_locations[key]["substation"].to_crs("EPSG:3857").copy()
    gdf_s = interested_locations[key]["region"].to_crs("EPSG:3857").copy()


    print(f"{key} Agent包含 {len(gdf_a)} 个点")
    print(f"{key} Target包含 {len(gdf_t)} 个点")
    print(f"{key} Source包含 {len(gdf_s)} 个区域")

    print("对坐标数据进行标准化...")

    # 1. 从 GeoDataFrame 中提取坐标
    coords_a = np.array([[point.x, point.y] for point in gdf_a.centroid])
    coords_t = np.array([[point.x, point.y] for point in gdf_t.geometry])
    coords_s = np.array([[point.x, point.y] for point in gdf_s.centroid])

    # 2. 合并所有坐标以进行统一缩放
    #    这是关键一步，确保A和B在一个统一的坐标系下进行变换，保持其相对空间关系
    all_coords = np.vstack([coords_a, coords_t, coords_s])

    # 3. 创建并拟合 StandardScaler
    scaler = StandardScaler()
    scaler.fit(all_coords)

    # 4. 对A和B的坐标进行变换
    coords_a_scaled = scaler.transform(coords_a)
    coords_t_scaled = scaler.transform(coords_t)
    coords_s_scaled = scaler.transform(coords_s)

    # 5. 创建新的 GeoDataFrame 副本来存储变换后的坐标
    #    （或者直接更新 gdf_a 和 gdf_b）
    gdf_a_scaled = gdf_a.copy()
    gdf_t_scaled = gdf_t.copy()
    gdf_s_scaled = gdf_s.copy()

    gdf_a_scaled['geometry'] = [Point(x, y) for x, y in coords_a_scaled]
    gdf_t_scaled['geometry'] = [Point(x, y) for x, y in coords_t_scaled]
    gdf_s_scaled['geometry'] = [Point(x, y) for x, y in coords_s_scaled]

    print("标准化完成。")
    # 建立substation_idx索引
    grid_proj = gdf_a.to_crs("EPSG:3857").copy()
    substations_proj = gdf_t.to_crs("EPSG:3857").copy()
    substations_proj = substations_proj.reset_index(drop=True)
    joined_gdf = gpd.sjoin_nearest(grid_proj, substations_proj[['geometry']], how="left", distance_col=None)
    joined_gdf.rename(columns={'index_right': 'substation_idx'}, inplace=True)
    gdf_a_scaled['substation_idx'] = joined_gdf['substation_idx']
    print("建立substation_idx索引完成。")

    # final_features_a = preprocess_features(gdf_a[["landuse"]])
    final_features_a = preprocess_features(gdf_a, numerical_col_names_all, categorical_col_members_all)
    final_features_t = preprocess_features(gdf_t[[]])
    final_features_s = preprocess_features(gdf_s[["residential_percent", "commercial_percent", "industrial_percent", "agricultural_percent", "others_percent"]])
    # final_features_s = preprocess_features(gdf_s[[]])
    graph_data = prepare_hetero_graph_from_processed(gdf_s_scaled, gdf_a_scaled, gdf_t_scaled, final_features_s, final_features_a, relation_column='ITL3')
    torch.save(graph_data, graph_data_path)

    print(f"Graph data for {key} saved to {graph_data_path}.")

TLI3 Agent包含 311937 个点
TLI3 Target包含 39 个点
TLI3 Source包含 4 个区域
对坐标数据进行标准化...
标准化完成。
建立substation_idx索引完成。
模式：使用预定义的特征架构...
根据架构处理 5 个数值特征...
根据架构处理 1 个类别特征...
处理完成。最终特征维度: (311937, 10)
特征映射: {'lu_residential_prop': (0, 1, 'numerical'), 'lu_others_prop': (1, 2, 'numerical'), 'lu_agricultural_prop': (2, 3, 'numerical'), 'lu_industrial_prop': (3, 4, 'numerical'), 'lu_commercial_prop': (4, 5, 'numerical'), 'landuse': (5, 10, 'categorical')}
------------------------------
模式：自动检测特征...
找到 0 个数值列: []
找到 0 个类别列: []
警告：未找到任何特征列，返回空的特征矩阵。
处理完成。最终特征维度: (39, 0)
特征映射: {}
------------------------------
模式：自动检测特征...
找到 5 个数值列: ['residential_percent', 'commercial_percent', 'industrial_percent', 'agricultural_percent', 'others_percent']
找到 0 个类别列: []
处理完成。最终特征维度: (4, 5)
特征映射: {'residential_percent': (0, 1, 'numerical'), 'commercial_percent': (1, 2, 'numerical'), 'industrial_percent': (2, 3, 'numerical'), 'agricultural_percent': (3, 4, 'numerical'), 'others_percent': (4, 5, 'numerical')}
---------------